# **UNIVERSIDADE FEDERAL DO CEARÁ**
---
Disciplina: Introdução à análise em Big Data

---

Professor: Luiz Alexandre

---

Alunos:
1.   Júlio César Gama Feitosa Freitas - 583956
2.   Vitória Freire Rocha Teixeira de Oliveira - 587661

---
Data: 13/09/2026

# 🧪 Lab 1 — HDFS: estrutura, upload e replicação

## 🎯 Objetivo

Criar a estrutura de pastas `raw/bronze/silver/gold` e subir os 3 datasets, confirmando a replicação 3× (rota cluster) ou organizando o equivalente local (rota sem admin).

> **Importante:** Como o Colab está sendo utilizado (Rota B), a replicação deste notebook é apenas uma **simulação local**.


## Configuração inicial

In [40]:
# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================
import os
import shutil
import hashlib
import pandas as pd

BASE = "../content/bigdata/"

# Deixa o notebook reproduzível
if os.path.exists(BASE):
    shutil.rmtree(BASE)

print("✓ Ambiente inicializado")


✓ Ambiente inicializado


## Upload dos datasets

Execute a célula abaixo e selecione os três arquivos:
- `customers_synthetic.csv`
- `transactions_synthetic.csv`
- `fraud_labels.csv`


In [41]:
# ============================================================
# 2. UPLOAD DOS DATASETS
# ============================================================

# Acessa a biblioteca de importação de arquivos do google colab
# from google.colab import files

# # Faz o upload dos arquivos
# uploaded = files.upload()

# print("\nArquivos recebidos:")
# for name in uploaded:
#     print(f"✓ {name}")


In [42]:
# ============================================================
# 3. VALIDAR OS DATASETS
# ============================================================
# Verificação de integridade dos datasets

# Define uma lista com os nomes esperados dos arquivos de dataset
EXPECTED = [
    "../customers_synthetic.csv",
    "../transactions_synthetic.csv",
    "../fraud_labels.csv"
]

# Verifica se todos os arquivos esperados existem no sistema de arquivos
missing = [f for f in EXPECTED if not os.path.exists(f)]

# Se algum arquivo estiver faltando, levanta um erro para interromper a execução
if missing:
    raise FileNotFoundError(
        "Arquivos ausentes: " + ", ".join(missing)
    )

print("Todos os datasets esperados foram encontrados.\n")

# Cada arquivo encontrado é lido como um DataFrame em pandas
# Imprime informações básicas
for filename in EXPECTED:
    df = pd.read_csv(filename)
    print(
        f"{filename:30} "
        f"{len(df):>8,} registros | "
        f"{os.path.getsize(filename):>10,} bytes"
    )


Todos os datasets esperados foram encontrados.

../customers_synthetic.csv        9,993 registros |    842,451 bytes
../transactions_synthetic.csv   100,000 registros |  8,139,648 bytes
../fraud_labels.csv               4,833 registros |    218,900 bytes


## Passo 1 — Criar a estrutura de camadas

In [43]:
# ============================================================
# 4. CRIAR ESTRUTURA DE CAMADAS
# ============================================================
# Criação da estrutura de diretórios para as camadas de dados (raw, bronze, silver, gold)

# Define uma lista dos diretórios e subdiretórios a serem criados
directories = [
    "raw/customers",
    "raw/transactions",
    "raw/fraud_labels",
    "bronze",
    "silver",
    "gold"
]

for directory in directories:
    os.makedirs(os.path.join(BASE, directory), exist_ok=True)

print("Estrutura criada:\n")
for directory in directories:
    print(f"✓ {BASE}/{directory}")


Estrutura criada:

✓ ../content/bigdata//raw/customers
✓ ../content/bigdata//raw/transactions
✓ ../content/bigdata//raw/fraud_labels
✓ ../content/bigdata//bronze
✓ ../content/bigdata//silver
✓ ../content/bigdata//gold


## Passo 2 — Copiar os datasets para dentro da estrutura

In [44]:
# ============================================================
# 5. COPIAR DATASETS PARA RAW
# ============================================================
# Cópia dos datasets originais para a camada 'raw' e organização dos subdiretórios específicos

# Define um dicionário que mapeia cada nome de arquivo de dataset para o seu diretório de destino na camada 'raw'.
destinations = {
    "../customers_synthetic.csv": f"../{BASE}raw/customers/customers_synthetic.csv",
    "../transactions_synthetic.csv": f"../{BASE}raw/transactions/transactions_synthetic.csv",
    "../fraud_labels.csv": f"../{BASE}raw/fraud_labels/fraud_labels.csv"
}

for filename, destination in destinations.items():
    shutil.copy2(
        filename,
        os.path.join(BASE, destination, filename)
    )
    print(f"✓ {filename} → {destination}/")


✓ ../customers_synthetic.csv → ../../content/bigdata/raw/customers/customers_synthetic.csv/
✓ ../transactions_synthetic.csv → ../../content/bigdata/raw/transactions/transactions_synthetic.csv/
✓ ../fraud_labels.csv → ../../content/bigdata/raw/fraud_labels/fraud_labels.csv/


## Passo 3 — Conferir a estrutura

In [45]:
# ============================================================
# 6. CONFERIR A ESTRUTURA
# ============================================================
# Exibição da estrutura de diretórios criada, com os arquivos contidos neles

# Usa os.walk para iterar por todos os diretórios e subdiretórios a partir do BASE
# Para cada item, calcula o nível de indentação e imprime o nome do diretório
for root, dirs, file_names in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    # Imprime os nomes dos arquivos dentro de cada diretório, também com indentação
    for filename in sorted(file_names):
        print(f"{indent}  └── {filename}")


/
bronze/
gold/
raw/
  customers/
    └── customers_synthetic.csv
  fraud_labels/
    └── fraud_labels.csv
  transactions/
    └── transactions_synthetic.csv
silver/


## Passo 5 — Contar linhas com Python

In [46]:
# ============================================================
# 7. CONTAGEM DE REGISTROS
# ============================================================
# Leitura dos datasets da camada 'raw' e contagem de registros em cada um.

# Define um dicionário que mapeia nomes descritivos (chaves)
paths = {
    "clientes": os.path.join(
        BASE, "raw/customers/customers_synthetic.csv"
    ),
    "transações": os.path.join(
        BASE, "raw/transactions/transactions_synthetic.csv"
    ),
    "fraud_labels": os.path.join(
        BASE, "raw/fraud_labels/fraud_labels.csv"
    )
}

# Inicializa um dicionário vazio para armazenar as contagens de registros de cada dataset
counts = {}

# Leitura iterativa de cada arquivo CSV, contando suas linhas e armazenando o resultado
for label, path in paths.items():
    df = pd.read_csv(path)
    counts[label] = len(df)
    print(f"{label:15}: {len(df):,}")


clientes       : 9,993
transações     : 100,000
fraud_labels   : 4,833


## Passo 4 — Simular a réplica (só para visualizar o conceito)

In [47]:
# ============================================================
# 8. SIMULAR REPLICAÇÃO 3×
# ============================================================
# Simulação da replicação de um dataset, criando três cópias do arquivo de transações

# Define o arquivo de origem para a simulação
source = paths["transações"]
# Cria um diretório temporário para armazenar as réplicas
replica_dir = os.path.join(BASE, "_replica_demo")
os.makedirs(replica_dir, exist_ok=True)

# Loop para criar 3 cópias do arquivo de origem
for i in range(1, 4):
    # Define o nome e caminho do arquivo de destino para cada cópia
    destination = os.path.join(
        replica_dir, f"copia_{i}.csv"
    )
    # Copia o arquivo de origem para o destino
    shutil.copy2(source, destination)

print("Réplicas criadas:\n")

# Imprime o nome e tamanho de cada réplica criada para verificação
for filename in sorted(os.listdir(replica_dir)):
    path = os.path.join(replica_dir, filename)
    print(
        f"✓ {filename:12} "
        f"{os.path.getsize(path):,} bytes"
    )


Réplicas criadas:

✓ copia_1.csv  8,139,648 bytes
✓ copia_2.csv  8,139,648 bytes
✓ copia_3.csv  8,139,648 bytes


In [48]:
# ============================================================
# 9. VERIFICAR INTEGRIDADE COM SHA-256
# ============================================================
# Verificação da integridade das cópias de arquivos usando hashes SHA-256

# Função para calcular o hash SHA-256 de um arquivo
def sha256(path):
    h = hashlib.sha256()
    # Lê o arquivo em blocos para lidar com arquivos grandes
    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()

# Calcula o hash do arquivo original
source_hash = sha256(source)

print("Hash do arquivo original:")
print(source_hash)

print("\nHashes das réplicas:")

# Flag para verificar se todas as réplicas são idênticas
all_equal = True

# Itera sobre as cópias para calcular e comparar seus hashes
for i in range(1, 4):
    path = os.path.join(
        replica_dir, f"copia_{i}.csv"
    )
    current_hash = sha256(path)
    # Compara o hash da cópia com o hash do original
    same = current_hash == source_hash
    all_equal = all_equal and same

    # Imprime o hash da cópia e um indicador de igualdade
    print(
        f"copia_{i}.csv: "
        f"{current_hash} "
        f"{'✓' if same else '✗'}"
    )

# Imprime uma mensagem final indicando se todas as réplicas são idênticas
print(
    "\n✓ As três réplicas são idênticas."
    if all_equal
    else "\n✗ Há diferenças entre as réplicas."
)


Hash do arquivo original:
55f45648d22b684e242f0e097d2f8d793638abdda0987bd87b09ef50ce816366

Hashes das réplicas:
copia_1.csv: 55f45648d22b684e242f0e097d2f8d793638abdda0987bd87b09ef50ce816366 ✓
copia_2.csv: 55f45648d22b684e242f0e097d2f8d793638abdda0987bd87b09ef50ce816366 ✓
copia_3.csv: 55f45648d22b684e242f0e097d2f8d793638abdda0987bd87b09ef50ce816366 ✓

✓ As três réplicas são idênticas.


In [49]:
# ============================================================
# 10. INSPEÇÃO RÁPIDA DOS DATASETS
# ============================================================
# Realiza uma inspeção rápida dos datasets da camada 'raw', exibindo informações básicas e uma amostra inicial

# Itera sobre cada dataset
for label, path in paths.items():
    # Lê o arquivo CSV para um DataFrame pandas
    df = pd.read_csv(path)

    print("=" * 70)
    # Imprime o nome do dataset em maiúsculas
    print(label.upper())
    # Imprime o número de linhas e colunas
    print(
        f"Linhas: {len(df):,} | "
        f"Colunas: {len(df.columns)}"
    )
    # Imprime os nomes das colunas
    print("Colunas:", ", ".join(df.columns))
    print()

    # Exibe as 3 primeiras linhas do DF
    display(df.head(3))


CLIENTES
Linhas: 9,993 | Colunas: 7
Colunas: customer_id, name, cpf, email, segment, credit_score, created_at



,customer_id,name,cpf,email,segment,credit_score,created_at
0,1,Ana Laura Campos,943.065.218-42,igor46@example.com,Premium,426,2026-07-03
1,2,Mariah Caldeira,586.237.094-38,jose48@example.com,High-Risk,481,2026-06-17
2,3,Kevin Cavalcante,530.629.814-15,theoda-costa@example.org,Standard,708,2025-11-06


TRANSAÇÕES
Linhas: 100,000 | Colunas: 8
Colunas: transaction_id, customer_id, amount, transaction_type, timestamp, status, risk_score, is_fraud



,transaction_id,customer_id,amount,transaction_type,timestamp,status,risk_score,is_fraud
0,9157,4627,156.683351,pagamento,2023-01-01,approved,89.859250,False
1,46882,9378,61.694980,compra,2023-01-01,approved,64.229229,False
2,91656,1385,41.844959,transferencia,2023-01-01,approved,0.183873,False


FRAUD_LABELS
Linhas: 4,833 | Colunas: 5
Colunas: transaction_id, customer_id, amount, date, label_fraud



,transaction_id,customer_id,amount,date,label_fraud
0,23,4569,1138.133971,2023-09-28,False
1,31,1943,394.826704,2023-05-24,False
2,38,6164,106.672531,2024-09-26,False


## ✅ Checkpoint

In [50]:
# ============================================================
# 11. CHECKPOINT AUTOMÁTICO
# ============================================================
# Verificação automática das etapas concluídas no notebook

# Converte a lista de diretórios esperados para caminhos absolutos
required_dirs = [
    os.path.join(BASE, d)
    for d in directories
]

# Verifica se todos os diretórios necessários foram criados
dirs_ok = all(
    os.path.isdir(d)
    for d in required_dirs
)

# Verifica se os arquivos dos datasets estão presentes nos caminhos esperados
files_ok = all(
    os.path.isfile(p)
    for p in paths.values()
)

# Verifica se a simulação de replicação foi bem-sucedida (3 cópias idênticas)
replicas_ok = (
    all_equal
    and len(os.listdir(replica_dir)) == 3
)

print("=" * 70)
print("CHECKPOINT — LAB 1")
print("=" * 70)

# Imprime o status da criação da estrutura de diretórios
print(
    f"[{'✓' if dirs_ok else '✗'}] "
    "Estrutura raw/bronze/silver/gold criada"
)

# Imprime o status da presença dos datasets na camada 'raw'
print(
    f"[{'✓' if files_ok else '✗'}] "
    "Os 3 datasets estão em raw"
)

# Imprime o status da simulação de replicação
print(
    f"[{'✓' if replicas_ok else '✗'}] "
    "Replicação local simulada em 3 cópias"
)

# Imprime as contagens de registros para cada dataset (apenas para confirmação)
print(f"[✓] Clientes:      {counts['clientes']:,}")
print(f"[✓] Transações:    {counts['transações']:,}")
print(f"[✓] Fraud labels:  {counts['fraud_labels']:,}")

print("=" * 70)
print(
    "\nOBSERVAÇÃO: a replicação 3× acima é uma "
    "simulação local, não HDFS distribuído."
)


CHECKPOINT — LAB 1
[✓] Estrutura raw/bronze/silver/gold criada
[✓] Os 3 datasets estão em raw
[✓] Replicação local simulada em 3 cópias
[✓] Clientes:      9,993
[✓] Transações:    100,000
[✓] Fraud labels:  4,833

OBSERVAÇÃO: a replicação 3× acima é uma simulação local, não HDFS distribuído.
